# 回帰Active Learning：獲得関数と観測更新の実践ガイド

回帰モデルで**どの入力を次に測れば予測を改善できるか**を、`qPosteriorVariance`・EPIG・`qNegIntegratedPosteriorVariance`・`qStraddle`で比較します。

学習 → 不確実性の可視化 → 候補最適化 → 観測 → 再学習 → 予測変化の確認まで実行します。獲得関数はBoTorchとrobotorchanの既存公開APIを利用します。

関連理論: [Active Learning](../../docs/theory/acquisition/09_active_learning.md) / [Level-set](../../docs/theory/acquisition/10_level_set.md)


## 1. Imports and reproducibility

In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.acquisition.active_learning import qNegIntegratedPosteriorVariance
from botorch.optim import optimize_acqf

from robotorchan.acquisition import (
    ExpectedPredictiveInformationGain,
    qPosteriorVariance,
    qStraddle,
)
from robotorchan.models import SingleTaskGP

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

### BOとActive Learningの違い

**ベイズ最適化（BO）** は良い目的関数値を見つけるために次の実験条件を選びます。一方、**Active Learning（AL）** はモデルがまだ十分に学べていない場所を測り、予測や境界の理解を改善します。

ここでは応答 `f(x)` の最大値は追いません。観測が少ない回帰GPの予測分布を改善することが目的です。獲得関数は「次にどこを測るか」の判断基準であり、予測モデルそのものではありません。

## 2. Sparse regression observations

function learningが目的なので、objectiveの最大値はdecision criterionに使いません。

In [ ]:
def response(X: torch.Tensor) -> torch.Tensor:
    return torch.sin(2 * torch.pi * X) + 0.2 * X


train_X = torch.tensor([[0.05], [0.22], [0.48], [0.72], [0.95]])
train_Y = response(train_X)
bounds = torch.tensor([[0.0], [1.0]])

model = SingleTaskGP(train_X, train_Y)
model.fit()
model.eval()

## 3. Posterior uncertainty

In [ ]:
grid = torch.linspace(0.0, 1.0, 300).unsqueeze(-1)
with torch.no_grad():
    posterior = model.posterior(grid)
mean = posterior.mean.squeeze(-1)
std = posterior.variance.clamp_min(0.0).sqrt().squeeze(-1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(grid.squeeze(-1), mean, label="posterior mean")
ax.fill_between(grid.squeeze(-1), mean - 2 * std, mean + 2 * std, alpha=0.2)
ax.scatter(train_X.squeeze(-1), train_Y.squeeze(-1), label="observations")
ax.legend()
plt.show()

### 事後平均と事後標準偏差を読む

GPの事後平均は現在の最も代表的な予測、標準偏差はその周辺の不確実性を表します。**平均が大きい場所**と**不確実性が大きい場所**は別です。

図の帯は平均±2標準偏差です。これは潜在関数の不確実性の目安であり、観測ノイズを含む予測区間と必ずしも同じではありません。

## 4. Pointwise function learning: qPosteriorVariance

`qPosteriorVariance`はcandidate自身のmarginal epistemic uncertaintyを基準にするq=1 criterionです。

In [ ]:
variance_acq = qPosteriorVariance(model)
variance_candidate, variance_value = optimize_acqf(
    acq_function=variance_acq,
    bounds=bounds,
    q=1,
    num_restarts=4,
    raw_samples=64,
)
print("variance candidate:", variance_candidate)
print("variance score:", variance_value)

### Variance最大化はどの点を選ぶか

`qPosteriorVariance` は候補自身の事後分散を大きくする条件を選びます。既知の観測点から離れた場所が選ばれることがありますが、**予測を改善したい領域が別にある場合**は、その領域への効果を直接評価するEPIGやqNIPVが適しています。

## 5. Prediction-oriented learning: EPIG

EPIGはcandidate自身が不確実かではなく、candidate観測が**target inputsでのpredictions**についてどれだけ情報を与えるかを評価します。ここではdeploymentが中央領域に集中すると仮定します。

In [ ]:
target_X = torch.linspace(0.35, 0.65, 31).unsqueeze(-1)
epig = ExpectedPredictiveInformationGain(
    model,
    target_X,
    observation_noise=1e-4,
)
epig_candidate, epig_value = optimize_acqf(
    acq_function=epig,
    bounds=bounds,
    q=1,
    num_restarts=4,
    raw_samples=64,
)
print("EPIG candidate:", epig_candidate)
print("EPIG score:", epig_value)

### EPIGのターゲット集合

`target_X` は将来の運用で予測精度を上げたい入力の集合です。今回の例では中央領域 `0.35〜0.65` を重視します。

候補自身の分散が最大でなくても、中央領域の予測に大きな影響を与える観測ならEPIGが高くなる可能性があります。**観測候補の位置と予測対象の位置は一致する必要がありません**。

## 6. Batch target-region learning: qNIPV

BoTorch native `qNegIntegratedPosteriorVariance`はintegration points全体で、batch観測後に残るposterior varianceを評価します。pointwise variance上位2点を取る方法ではありません。

In [ ]:
integration_X = torch.linspace(0.0, 1.0, 41).unsqueeze(-1)
nipv = qNegIntegratedPosteriorVariance(model=model, mc_points=integration_X)
nipv_candidates, nipv_value = optimize_acqf(
    acq_function=nipv,
    bounds=bounds,
    q=2,
    num_restarts=3,
    raw_samples=32,
)
assert nipv_candidates.shape == torch.Size([2, 1])
print("qNIPV joint candidates:", nipv_candidates.squeeze(-1))
print("qNIPV value:", nipv_value)

### qNIPVの2点同時提案

`qNegIntegratedPosteriorVariance` は、指定した積分点における**観測後の事後分散**を小さくするように2点を選びます。2点の組合せを同時評価するため、単一点分散の上位2点とは異なります。

ここでの `integration_X` は積分領域の近似点であり、実際に観測した訓練データではありません。

## 7. Boundary learning is a different AL objective

function全体ではなく`f(x)=target`の境界を学ぶならqStraddleを使えます。uncertaintyだけでなくposterior meanのtarget proximityもcriterionに入ります。

In [ ]:
target_level = 0.0
straddle = qStraddle(model, target=target_level, beta=1.96)
boundary_candidate, boundary_value = optimize_acqf(
    acq_function=straddle,
    bounds=bounds,
    q=1,
    num_restarts=4,
    raw_samples=64,
)
print("qStraddle candidate:", boundary_candidate)
print("qStraddle score:", boundary_value)

### qStraddleは閾値近傍を学ぶ

製造プロセスでは、品質がある閾値を上回るかどうかの**境界**が重要な場合があります。`qStraddle` は平均が閾値に近く、不確実性もある点を重視します。

閾値 `target_level=0` は回帰応答の値です。分類ラベル0を意味するわけではありません。

## 8. Candidate locations reflect different learning targets

acquisition valueの絶対値はcriterion間で比較しません。Variance、EPIG、qNIPV、qStraddleは学習したいquantity自体が異なります。

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(grid.squeeze(-1), response(grid).squeeze(-1), label="true response")
ax.scatter(train_X.squeeze(-1), train_Y.squeeze(-1), label="observations")
for name, candidate in {
    "Variance": variance_candidate,
    "EPIG": epig_candidate,
    "qStraddle": boundary_candidate,
}.items():
    ax.scatter(
        candidate.item(),
        response(candidate).item(),
        marker="x",
        s=80,
        label=name,
    )
for index, candidate in enumerate(nipv_candidates):
    ax.scatter(
        candidate.item(),
        response(candidate.unsqueeze(0)).item(),
        marker="+",
        s=90,
        label="qNIPV batch" if index == 0 else None,
    )
ax.legend(ncol=2)
plt.show()

## 9. Observe one AL candidate and update the posterior

代表としてprediction-orientedなEPIG candidateを観測し、dataへ追加してrefitします。

### EPIGで選んだ候補を観測する

実際のALでは候補を提案した後に実験・測定を行い、初めて新しい応答値を取得します。ここでは人工関数 `response` を測定器の代わりに使います。

観測を追加したら**新しいGPを学習**し、中央領域の予測分散がどう変わったかを確認します。単一の小規模実験なので、分散低下だけで実務上の性能改善を保証するものではありません。

In [ ]:
new_Y = response(epig_candidate)
updated_X = torch.cat([train_X, epig_candidate], dim=0)
updated_Y = torch.cat([train_Y, new_Y], dim=0)

updated_model = SingleTaskGP(updated_X, updated_Y)
updated_model.fit()
updated_model.eval()

with torch.no_grad():
    before = model.posterior(target_X).variance.mean()
    after = updated_model.posterior(target_X).variance.mean()
print("mean target variance before:", before.item())
print("mean target variance after:", after.item())

### 更新前後の予測を可視化する

EPIGが提案した点を観測する前後で、事後平均と不確実性の帯を比較します。中央領域 `target_X` を背景に示し、どの範囲の予測改善を狙っていたか確認します。

In [ ]:
with torch.no_grad():
    updated_posterior = updated_model.posterior(grid)
    updated_mean = updated_posterior.mean.squeeze(-1)
    updated_std = updated_posterior.variance.clamp_min(0.0).sqrt().squeeze(-1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True, sharey=True)
for ax, center, uncertainty, title in (
    (axes[0], mean, std, "Before observation"),
    (axes[1], updated_mean, updated_std, "After observation and fit"),
):
    ax.axvspan(0.35, 0.65, alpha=0.08, label="EPIG target region")
    ax.plot(grid.squeeze(-1), response(grid).squeeze(-1), label="True response")
    ax.plot(grid.squeeze(-1), center.detach(), label="Posterior mean")
    ax.fill_between(grid.squeeze(-1), (center - 2 * uncertainty).detach(), (center + 2 * uncertainty).detach(), alpha=0.2)
    ax.scatter(train_X.squeeze(-1), train_Y.squeeze(-1), label="Initial observations")
    ax.scatter(epig_candidate.squeeze(-1), new_Y.squeeze(-1), marker="x", s=75, label="New observation")
    ax.set(xlabel="x", title=title)
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

# 再学習したモデルでEPIGを作り直し、次の観測点を提案
next_epig = ExpectedPredictiveInformationGain(
    updated_model, target_X, observation_noise=1e-4
)
next_candidate, next_value = optimize_acqf(
    acq_function=next_epig, bounds=bounds, q=1,
    num_restarts=4, raw_samples=64,
)
print("次のEPIG提案:", next_candidate, "score:", next_value)
assert updated_X.shape[0] == train_X.shape[0] + 1
assert next_candidate.shape == (1, 1)

## 10. Decision-purpose map

| Criterion | Learning target | Candidate semantics |
| --- | --- | --- |
| qPosteriorVariance | candidateのlatent function | q=1 marginal uncertainty |
| EPIG | target-set predictions | q=1 predictive information gain |
| qNIPV | integration region | joint batch variance reduction |
| qStraddle | specified response boundary | q=1 uncertainty + target proximity |

BOのEI/UCBのように「良いobjective valueを見つける」ことは、このNotebookの最終目的ではありません。

## 11. Important boundaries

- predictive varianceとreducible epistemic uncertaintyを無条件に同一視しません。
- EPIGのcurrent contractはq=1、single-output、joint Gaussian posterior、finite `target_X`です。
- current `qPosteriorVariance/Std` runtimeはensemble posteriorをrejectします。
- qNIPVはBoTorch native pathを直接使い、robotorchan wrapperを追加しません。
- qStraddleのpointwise上位q点はjoint batch level-set acquisitionではありません。
- classification-specific entropy/BALD/margin acquisitionは現在のregression workflowへ混ぜません。
- MESはoptimum valueについてのinformation gainであり、prediction-oriented EPIGとはinformation targetが違います。

## 12. Related components

- `docs/optimization/regression_active_learning.md`: regression AL runtime guide
- `docs/optimization/epig.md`: EPIG contract
- `docs/theory/acquisition/05_information_theoretic.md`: BO側information target
- `docs/theory/acquisition/10_level_set.md`: boundary learning

Active Learningでは最初に「information / uncertainty about what?」を決めてからacquisitionを選びます。

## 13. 既存APIの対応と使い分け

| 学習したいもの | 獲得関数 | 候補提案 |
|---|---|---|
| 候補点の潜在関数の不確実性 | `qPosteriorVariance` | `optimize_acqf(q=1)` |
| 運用対象の予測情報 | `ExpectedPredictiveInformationGain` | `optimize_acqf(q=1)` |
| 領域全体の事後分散低減 | `qNegIntegratedPosteriorVariance` | `optimize_acqf(q=2)` |
| 指定した応答閾値の境界 | `qStraddle` | `optimize_acqf(q=1)` |
| GP学習・再学習 | `SingleTaskGP.fit()` | 観測追加後に実行 |

**まとめ：** どの獲得関数が優れているかは、何を学びたいかで変わります。関数全体、運用対象領域、閾値境界のどれを改善したいかを先に決め、目的に対応する既存の獲得関数を選んでください。